# Docking and scoring notebook

Protocol for docking ligands into a target protein and rescoring the poses:

1. **RDKit** generates a diverse, low-energy set of conformations for each ligand.
2. The binding site is defined either from a **protein–ligand complex PDB** (the ligand is extracted for you) or from **prepared files** (protein without ligand + extracted ligand).
3. **smina** (https://sourceforge.net/projects/smina/) docks the conformers into the site (rigid or flexible side chains).
4. **AEV-PLIG** (Valsson *et al.*, *Commun. Chem.* 2025) predicts the binding affinity (pK) of every pose.

The example is the SARS-CoV-2 main protease (Mpro) fragment structure 5RHD.

**Kernel:** `docking_scoring` conda env. One-off setup, in a terminal in this folder (see README.md for details and for installing smina):

```bash
conda env create -f environment.yml
conda run -n docking_scoring python -m ipykernel install --user --name docking_scoring --display-name "Python (docking_scoring)"
conda run -n docking_scoring python aev_plig/download_models.py
```

**Output files** are written next to the input files they come from; all prediction results go to a `predictions` folder next to the docked SDF.

Widgets: each step has a panel with a **Run** button. Set `AUTO_RUN = True` in the next cell to run every step with the default settings when executing all cells.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from rdkit import Chem
from rdkit.Chem import Draw

import docking_tools as dt

%matplotlib inline

AUTO_RUN = os.environ.get("DOCKING_AUTORUN") == "1"   # set True to run all steps with defaults
dt.PROTEIN_COLOURING = "rasmol"   # protein cartoons: "rasmol" (secondary structure), "rainbow" or "grey"

print(dt.cpu_note())
dt.library_versions()

## 1. Ligands to dock
Set the path of the SDF file of structures you want to dock. The name must be on the first line of each record, e.g.

<code>AEM 10028511
  MOE2019           2D
</code>

Conformers, docked poses and predictions will be written to the **same folder** as this file.

In [ ]:
LIGAND_SDF = Path("example/Fordocking.sdf").resolve()   # input structures to dock

input_mols = dt.load_sdf(LIGAND_SDF)   # warns about unreadable or unnamed records
print(f"{len(input_mols)} molecules read from {LIGAND_SDF}")

### Check ligand names
smina keeps only the molecule name, and the results (best pose per ligand, PDB export, plots) are grouped by name, so **different structures must not share a name**. If duplicates are found you can either use an **identifier field** from the SDF (e.g. `ID`, `IDNUMBER`, `Record_ID` – offered only if every molecule has a unique value) as the ligand ID, or have **unique names assigned automatically** (`name`, `name_2`, `name_3` …). The original name is kept in the `original_name` property. The input SDF itself is not changed. Conformer generation is blocked until duplicates are resolved.

In [ ]:
name_check = dt.LigandNamePanel(input_mols)
display(name_check.widget())
if AUTO_RUN and not name_check.resolved:
    name_check.apply()   # default: first unique identifier field, otherwise automatic names

In [ ]:
# View as grid (re-run after applying a naming choice above)
Draw.MolsToGridImage([dt.to_2d(m) for m in input_mols], molsPerRow=4, subImgSize=(250, 200),
                     legends=[m.GetProp("_Name") for m in input_mols])

### Browse individual input structures
Use the slider (or Prev / Next) to step through the molecules.

In [ ]:
dt.molecule_browser(input_mols)

## 1b. Ionisation states with QupKake

[QupKake](https://github.com/Shualdon/QupKake) (Abarbanel & Hutchison, *J. Chem. Theory Comput.* 2024) predicts **micro-pKa** values by combining GFN2-xTB calculations with graph neural networks. Each ligand is neutralised, QupKake predicts the pKa of every acidic and basic site, and the ionisation state at the chosen pH is set:

* the **most basic site** is protonated if its pKa is above the pH, and the **most acidic site** is deprotonated if its pKa is below it (at most one of each; other sites that also cross the pH are listed in *also_crossing_pH*);
* **implausible basic sites are ignored** (on by default): QupKake sometimes reports sulphonamide N, amide N or aryl-amine N (e.g. the N-aryl nitrogen of an aryl-piperazine) as the most basic site, which would put the charge in the wrong place – these are listed in *ignored*.

The ligands in their predicted states are written to `<input>_pH7.4.sdf` next to the input, and – with *Dock the ligands in their predicted ionisation states* ticked – that file is used for conformer generation and docking, so all later outputs are named `<input>_pH7.4_…` and never overwrite the results for the input states. The same prediction is made for the ligand from the PDB (section 3), which is used as the reference in the final table.

QupKake runs in its own conda environment (`qupkake`, with xtb 6.4.1); if it is not installed the input states are used. **Re-run the conformer cell after running QupKake.**

In [ ]:
prot_panel = dt.ProtonationPanel(input_mols, LIGAND_SDF)
display(prot_panel.widget())
if AUTO_RUN and dt.qupkake_available():
    prot_panel.run()

## 2. Conformation generation

*From the RDKit documentation: conformer generation is a difficult and subtle task.* smina rotates around torsions during docking, but it does not flip rings or change amide geometry, so supplying several ring/amide conformers helps.

**Are there better alternatives within RDKit than the original cell?** Yes. The original notebook called `EmbedMultipleConfs(m, numConfs=3)` with no pruning or minimisation, so it could return near-duplicate, unrelaxed geometries. This notebook uses:

| Step | What it does | Why |
|---|---|---|
| **srETKDGv3** (default) / ETKDGv3 / ETKDG / ETDG | distance geometry + experimental torsion preferences | v3 has improved torsion libraries and macrocycle handling; the *sr* variant adds small-ring torsion preferences (better 3–7 membered ring puckers) |
| `pruneRmsThresh` | discards embeddings closer than the RMS threshold | diversity rather than duplicates |
| **MMFF94** optimisation (UFF fallback) | relaxes bond lengths/angles and intramolecular clashes | cleaner starting geometry for docking |
| Energy window | drops conformers more than *n* kcal/mol above the minimum | removes strained geometries |
| Post-optimisation RMS de-duplication (symmetry-aware heavy-atom RMSD) | minimisation often collapses conformers onto each other | keeps only distinct ones |
| Optional stereoisomer enumeration | expands unassigned stereocentres | avoids docking an arbitrary isomer |

Other RDKit options not used here: Torsion-Fingerprint/Butina clustering for very large ensembles, and `rdDistGeom.EmbedMultipleConfs` with `useRandomCoords=True` (used automatically as a fallback when embedding fails). For a single rigid molecule set *Conformers to embed* = 1.

The output file is `<input>_confs.sdf` in the input folder.

In [ ]:
DOCK_MOLS, DOCK_SDF = prot_panel.docking_input()   # predicted ionisation states if QupKake was run
print(f"Docking {len(DOCK_MOLS)} ligands from {DOCK_SDF.name}")
conf_panel = dt.ConformerPanel(DOCK_MOLS, DOCK_SDF)
display(conf_panel.widget())
if AUTO_RUN:
    conf_panel.run()

In [ ]:
# Check the conformer file (re-run this cell after generating)
CONF_SDF = conf_panel.out_path
if CONF_SDF.exists():
    confs = dt.load_sdf(CONF_SDF)
    print(f"{len(confs)} conformers in {CONF_SDF}")
    dt.molecule_browser(confs)

## 3. Protein and binding site

Two options:

* **Complex structure** – a protein–ligand complex in **PDB or mmCIF** format (`.pdb`, `.cif`, `.mmcif`, optionally `.gz`; mmCIF is now the PDB's standard format, e.g. `5RHD.cif` from RCSB). All **ligands** are listed with their chemical names and the **largest is selected automatically** – choose another from the list if the file contains several ligands. Waters, solvents and crystallisation additives (e.g. DMSO, glycerol, sulfate), ions and terminal caps are hidden; tick *Also list solvents, ions, additives and caps* to show them. Only the first alternate location (A) is used.
  The ligand's bond orders come from (in order): an optional **SMILES template** (its charges are kept, so it can set the protonation state); the file's **hydrogens** and formal charges; the ligand's **PDB Chemical Component Dictionary** entry from RCSB (for files without hydrogens such as RCSB X-ray structures – neutral form); or **Open Babel** perception.
  Two files are written next to the input: `<stem>_protein.pdb` (protein only, no ligand/other HETs) and `<stem>_ligand_<RES><chain><num>.pdb/.sdf`. For mmCIF input `<stem>` ends in `_cif` (e.g. `5RHD_cif_protein.pdb`) so it never overwrites files made from a PDB of the same entry, which may use a different coordinate frame.
* **Prepared files** – as in the original notebook: a protein PDB with no ligand plus the ligand extracted from the binding site (PDB).

> The example uses the MOE-prepared `5RHD.pdb` (hydrogens added, piperazine protonated). To use the RCSB file instead, set the path to `example/5RHD.cif`; its ligand (US7) has no hydrogens and is taken as the neutral molecule – paste `CS(=O)(=O)c1ccc(N2CC[NH2+]CC2)cc1` as the SMILES template for the protonated form.

Press **Use this binding site** after changing anything.

In [ ]:
site = dt.BindingSiteSetup(complex_pdb=Path("example/5RHD.pdb").resolve(),
                           protein_pdb=Path("example/proteinNoligand.pdb").resolve(),
                           ligand_pdb=Path("example/LigandOnly.pdb").resolve())
display(site.widget())
site.apply();   # applies the defaults; press the button to re-apply after changes

In [ ]:
# Ionisation state of the ligand from the PDB (crystal coordinates kept), with the same QupKake settings
REF_MOL = site.ligand_mol
if site.ligand_mol is not None and prot_panel.use.value and prot_panel.ionised is not None:
    REF_MOL, ref_table = dt.ionise_reference_ligand(site.ligand_mol, LIGAND_SDF.parent / "qupkake" / "pdb_ligand",
                                                    prot_panel.ph.value, prot_panel.filter.value)
    display(ref_table[["name", "changed", "input_charge", "predicted_charge", "protonated", "ignored", "smiles"]])
print("Reference ligand:", Chem.MolToSmiles(Chem.RemoveHs(REF_MOL)) if REF_MOL is not None else None)

### Images of the complex and the binding site
The protein is coloured by secondary structure with the **RasMol** scheme (helix magenta-red, sheet yellow, coil/turn white – drawn light grey so it shows on a white background); change `dt.PROTEIN_COLOURING` in the first code cell for rainbow or grey. Ray-traced with PyMOL and saved as `<stem>_complex.png` and `<stem>_site.png` next to the input structure. In the binding-site image residues within 5 Å of the ligand are shown as sticks, and polar contacts as orange dashes. Untick **Show binding-site side chains** to redraw the binding-site image without the side chains (and their polar contacts); the residues stay labelled.

In [ ]:
show_side_chains = dt.widgets.Checkbox(value=True, description="Show binding-site side chains",
                                       layout=dt.widgets.Layout(width="400px"))
image_out = dt.widgets.Output()

def _draw_images(_=None):
    global complex_png, site_png
    image_out.clear_output()
    with image_out:
        complex_png, site_png = dt.make_images(site.protein, site.ligand_ref, site.image_prefix,
                                               site_side_chains=show_side_chains.value)
        dt.show_images_side_by_side([complex_png, site_png],
                                    ["Protein with ligand bound", "Ligand in the binding site"])
        display(HTML(dt.colour_legend_html()))

show_side_chains.observe(_draw_images, names="value")
display(dt.widgets.VBox([show_side_chains, image_out]))
_draw_images()

## 4. Docking with smina

The SARS-CoV-2 main protease (Mpro) cleaves the viral polyprotein and is essential for viral replication (https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9283855/). The crystallised protein was the subject of a fragment screen at Diamond (https://www.diamond.ac.uk/industry/Case-Studies/Case-study-Fragment-Screening-to-fight-COVID-19.html); 5RHD is one of the resulting structures.

The panel is pre-filled with the settings from the original notebook (`--exhaustiveness 20 --seed 0 --autobox_ligand`) and `--cpu` set to the number of cores detected. Notes:

* **`--minimize`** performs only a local energy minimisation of the supplied coordinates (no search). Use it on poses that are already in the site, e.g. the crystal ligand SDF or a docked SDF — not on freshly generated conformers. The output defaults to `<input>_minimized.sdf`.
* **Docking mode** – *Rigid receptor* (default) or *Flexible side chains*. ⚠️ **Flexible docking is much slower** (typically 10–100× longer, rising steeply with the number of flexible residues). Residues can be chosen
  * **automatically** – every residue with an atom within a user-defined distance of the site-defining ligand (default 5 Å), or
  * **manually** – pick residues from the list (nearest first) and/or type extra ones (`A:41,A:145`).

  Gly, Ala and Pro are excluded (no side-chain torsions). smina then also writes `<output>_flexres.pdb` holding the flexed side chains for **every pose, in the same order as the docked SDF** – the viewer and AEV-PLIG use it to pair each ligand pose with its own receptor conformation.
* **Search-box check.** The box is the autobox ligand's extent (all atoms, hydrogens included) plus *autobox_add* on every side; smina only searches inside it and gives no warning if a ligand is too big. The panel compares each ligand's longest span (from the conformer file) with the box and warns when a ligand is *tight* (less than 2 Å to spare along the smallest box side) or *longer than the smallest box side*, suggesting an *autobox_add*. After docking, poses with a heavy atom within 1 Å of a box face are reported, and the results tables get `box_margin` (Å; negative = outside) and `at_box_edge` – a best pose at the edge means the box may be limiting that ligand.
* The docking log is written next to the output as `<output>.log`, together with a run record `<output>.run.json` (smina version, full command, seed, `--cpu` and all settings, platform, SHA-256 of the inputs) so every run is self-describing. With this smina build the same seed gives the same poses for any `--cpu` (see README, Reproducibility).

In [ ]:
panel = dt.SminaPanel(receptor=site.protein, autobox_ligand=site.ligand_pdb, ligands=CONF_SDF)
display(panel.widget())
if AUTO_RUN:
    panel.run()

## 5. View docking results
Poses are sorted by smina score (best first). Step through them with the slider; the ligand from the original PDB (reference) is shown with red carbons and the current pose with green carbons. Untick **Ligand from PDB (red)** to hide the reference ligand.

After **flexible docking** each pose is shown together with *its own* flexed side chains (magenta carbons), with the original crystal side-chain positions overlaid in thin grey (toggle). The receptor for each pose is rebuilt from the rigid protein plus that pose's model in `<output>_flexres.pdb`.

In [ ]:
# Re-run this cell after smina has finished
DOCKED_SDF = Path(panel.output.value)
poses, pose_table, FLEX_PDB = [], None, None
if DOCKED_SDF.exists():
    poses = dt.load_poses(DOCKED_SDF)
    FLEX_PDB = dt.flex_file_for(DOCKED_SDF)       # present only after flexible docking
    FLEX_PDB = FLEX_PDB if FLEX_PDB.exists() else None
    print(f"{len(poses)} poses in {DOCKED_SDF}")
    if FLEX_PDB:
        print(f"Flexible side chains: {FLEX_PDB}")
    pose_table = dt.results_browser(poses, site.protein, site.ligand_ref, flex_pdb=FLEX_PDB)
else:
    print(f"{DOCKED_SDF} not found - press 'Run smina' above, then re-run this cell")

In [ ]:
pose_table.head(10) if pose_table is not None else None

## 6. Binding-affinity prediction with AEV-PLIG

AEV-PLIG is a graph neural network scoring function (atomic environment vectors + protein–ligand interaction graphs). The model used is `model_GATv2Net_ligsim90_fep_benchmark` (ensemble of 10), copied with the prediction code into `aev_plig/` in this project. Predictions are **pK** values (higher = tighter binding); the spread across the 10 models is reported as `AEV_PLIG_pK_std`.

All results are exported to `predictions/` next to the docked SDF:

| File | Content |
|---|---|
| `poses/<docked>/pose_<i>.sdf` | one file per pose (AEV-PLIG input) |
| `poses/<docked>/pose_<i>_receptor.pdb` | flexible docking only: the receptor with that pose's flexed side chains (each pose is scored against its own receptor) |
| `<docked>_aevplig_input.csv` | AEV-PLIG input table |
| `<docked>_predictions.csv` | raw AEV-PLIG output (10 models + mean) |
| `<docked>_summary.csv` | name, pose, smina score, pK, pK std, best-pose flags |
| `<docked>_scored.sdf` | all poses annotated with pK |
| `<docked>_best_poses.sdf` | highest-pK pose per ligand |
| `<docked>_best_poses_table.csv` | best pose per ligand **plus the ligand from the original PDB** in its crystal pose: AEV-PLIG pK (also the red dashed line on the plot) and smina scores, the **box check** (`box_margin`, `at_box_edge`), plus **ligand efficiency** (`heavy_atoms`, `LE_AEV_PLIG` = 1.364 × pK / heavy atoms, `LE_smina` = −smina score / heavy atoms; kcal/mol per heavy atom) – `minimizedAffinity` after local minimisation in the site (comparable with the docked poses), `smina_score_only` for the unmodified crystal coordinates and `minimizedRMSD` (how far minimisation moved it) |
| `pdb_ligand/` | the PDB ligand as scored by smina, its minimised pose and the smina log |
| `<docked>_best_poses.pse` | PyMOL session of the best poses (section 7) |
| `best_poses_pdb/<docked>/<ligand>_<pose>_ligand.pdb` and `..._complex.pdb` | highest-pK pose per ligand in PDB format: ligand only, and protein–ligand complex (ligand = chain L, resname LIG) using that pose's own receptor (flexed side chains after flexible docking); REMARKs give smina score and pK |
| `<docked>_smina_vs_aevplig.png` | score comparison plot |
| copies of docked SDF, receptor, `_flexres.pdb` (flexible), reference ligand, conformers and images | for a self-contained record |

In [ ]:
predict_button = dt.widgets.Button(description="Run AEV-PLIG", button_style="success", icon="play")
predict_out = dt.widgets.Output()
summary = None

def _predict(_=None):
    global summary
    predict_out.clear_output()
    with predict_out:
        if not DOCKED_SDF.exists():
            print("No docked SDF yet - run smina and the 'View docking results' cell first")
            return
        extras = [CONF_SDF, site.ligand_pdb, site.ligand_ref, complex_png, site_png,
                  DOCKED_SDF.with_suffix(".log")]
        # smina scores for the ligand from the PDB in its crystal pose (same smina + scoring function)
        ref_smina = None
        if REF_MOL is not None:
            ref_smina = dt.smina_score_ligand(REF_MOL, site.protein,
                                              DOCKED_SDF.parent / "predictions" / "pdb_ligand",
                                              smina=panel.smina.value, scoring=panel.scoring_arg)
        summary = dt.run_aev_plig(DOCKED_SDF, site.protein, extra_files=extras, flex_pdb=FLEX_PDB,
                                  reference_mol=REF_MOL,     # also score the ligand from the PDB
                                  reference_smina=ref_smina)
        display(summary.head(10))

predict_button.on_click(_predict)
display(dt.widgets.VBox([predict_button, predict_out]))
if AUTO_RUN:
    _predict()

### Step through poses ranked by AEV-PLIG pK

In [ ]:
if summary is not None:
    dt.results_browser(poses, site.protein, site.ligand_ref, predictions=summary, sort_by="AEV_PLIG_pK",
                       flex_pdb=FLEX_PDB)

In [ ]:
# Best pose per ligand by AEV-PLIG, plus the ligand from the original PDB in its crystal pose.
# Ligand efficiency (kcal/mol per heavy atom): LE_AEV_PLIG = 1.364 x pK / heavy atoms; LE_smina = -smina score / heavy atoms
# (best poses are also exported as PDB: ligand only + protein-ligand complex)
if summary is not None:
    best_table = pd.read_csv(DOCKED_SDF.parent / "predictions" / f"{DOCKED_SDF.stem}_best_poses_table.csv")
    display(best_table.style.format({"minimizedAffinity": "{:.2f}", "AEV_PLIG_pK": "{:.2f}",
                                     "AEV_PLIG_pK_std": "{:.2f}", "smina_score_only": "{:.2f}",
                                     "minimizedRMSD": "{:.2f}", "LE_AEV_PLIG": "{:.3f}",
                                     "LE_smina": "{:.3f}", "box_margin": "{:.2f}"}, na_rep="–")
            .apply(lambda r: ["background-color: #fde2e2" if r["source"] == "ligand from PDB" else ""] * len(r),
                   axis=1).hide(axis="index"))
    best_pdb_dir = DOCKED_SDF.parent / "predictions" / "best_poses_pdb" / DOCKED_SDF.stem
    print(f"PDB files in {best_pdb_dir}:")
    for f in sorted(best_pdb_dir.glob("*.pdb")):
        print("  ", f.name)

## 7. Export best poses as a PyMOL session
Saves `predictions/<docked>_best_poses.pse` with the best AEV-PLIG pose of each ligand, ready to open in PyMOL:

* `receptor` – cartoon (RasMol secondary-structure colours) with binding-site side chains as light-blue sticks
* one group per ligand, ranked by pK (`01_<ligand>`, `02_<ligand>` …) containing the pose (`_pose`), its polar contacts (`_hbonds`) and, after flexible docking, that pose's flexed side chains (`_flexed`, magenta)
* `PDB_ligand` – the ligand from the original PDB (red carbons)

Only the top-ranked ligand is switched on initially (tick **Show all ligands** to enable every one); toggle groups in PyMOL's object panel. Each pose's title shows its name, pose id, pK and smina score.

In [ ]:
pse_all = dt.widgets.Checkbox(value=False, description="Show all ligands when the session opens")
pse_button = dt.widgets.Button(description="Export PyMOL session", button_style="success", icon="download")
pse_out = dt.widgets.Output()

def _export_pse(_=None):
    pse_out.clear_output()
    with pse_out:
        if summary is None:
            print("Run AEV-PLIG first (section 6)")
            return
        dt.export_pymol_session(DOCKED_SDF, site.protein, reference_ligand=site.ligand_ref,
                                enable="all" if pse_all.value else "top")

pse_button.on_click(_export_pse)
display(dt.widgets.VBox([pse_all, pse_button, pse_out]))
if AUTO_RUN:
    _export_pse()